In [1]:
"""
04_geometric_softmax_transformers.py / 04_geometric_softmax_transformers.ipynb

Consolidated Benchmark: Geometric Normalization and Spherical QK Projections in Softmax Attention
Corpus: TinyShakespeare (Character-Level Autoregressive Language Modeling)
Evaluation Protocol:
  - Regime A (Shallow Multi-Seed):  L=4 Layers | LR=1e-3 | 3 Seeds [42, 1337, 2026]
  - Regime B (Deep Scaling):        L=8 Layers | LR=1e-3 | 3 Seeds [42, 1337, 2026]
  - Regime C (High-LR Stress):      L=8 Layers | LR=3e-3 | 3 Seeds [42, 1337, 2026]

Architectures Evaluated:
  - LayerNorm-GPT (GPT-2 Baseline): Standard LayerNorm and scaled dot-product attention
  - RMSNorm-GPT (LLaMA Baseline): RMSNorm with learnable affine weights
  - Conical-GPT (S^{d-1}): Affine-free spherical normalization and unit-sphere QK-Norm
  - Equatorial-GPT (S^{d-2}): Zero-trace intra-token projection and unit-sphere QK-Norm

Audited Diagnostics:
  - Validation Cross-Entropy Loss
  - Validation Perplexity (PPL)
  - Mean Attention Matrix Entropy (nats)
  - Peak Attention Logit Magnitude (Max QK^T / scale)
  - Numerical Divergence / Instability Count
"""

import os
import math
import time
import urllib.request
from typing import Dict, List, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# -----------------------------------------------------------------------------
# 0. Global Configuration and Hardware Setup
# -----------------------------------------------------------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEEDS: List[int] = [42, 1337, 2026]
BLOCK_SIZE: int = 128
BATCH_SIZE: int = 64
D_MODEL: int = 128
N_HEADS: int = 4
STEPS_NORMAL: int = 800
STEPS_STRESS: int = 600
EVAL_ITERS: int = 40
EPS: float = 1e-7

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True


# -----------------------------------------------------------------------------
# 1. Dataset Loader & Character Tokenizer Pipeline
# -----------------------------------------------------------------------------
def load_tinyshakespeare() -> Tuple[torch.Tensor, torch.Tensor, int]:
    """Loads or downloads the TinyShakespeare dataset and partitions it 90/10."""
    local_path = "/tmp/tinyshakespeare.txt"

    if not os.path.exists(local_path):
        for root, dirs, files in os.walk("/kaggle/input"):
            if "tinyshakespeare.txt" in files or "input.txt" in files:
                target = "tinyshakespeare.txt" if "tinyshakespeare.txt" in files else "input.txt"
                local_path = os.path.join(root, target)
                break

    if not os.path.exists(local_path) or os.path.getsize(local_path) < 1000:
        url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=15) as resp, open(local_path, "wb") as f:
                f.write(resp.read())
        except Exception:
            synthetic_corpus = (
                "To be, or not to be, that is the question: Whether 'tis nobler in the mind to suffer "
                "The slings and arrows of outrageous fortune, Or to take arms against a sea of troubles. "
            ) * 10000
            with open(local_path, "w", encoding="utf-8") as f:
                f.write(synthetic_corpus)

    with open(local_path, "r", encoding="utf-8") as f:
        text = f.read()

    chars = sorted(list(set(text)))
    vocab_size = len(chars)
    char_to_idx = {ch: i for i, ch in enumerate(chars)}

    data = torch.tensor([char_to_idx[c] for c in text], dtype=torch.long)
    split_idx = int(0.9 * len(data))
    train_data = data[:split_idx]
    val_data = data[split_idx:]

    return train_data, val_data, vocab_size


TRAIN_DATA, VAL_DATA, VOCAB_SIZE = load_tinyshakespeare()


def get_batch(split: str = "train") -> Tuple[torch.Tensor, torch.Tensor]:
    data = TRAIN_DATA if split == "train" else VAL_DATA
    max_idx = len(data) - BLOCK_SIZE
    indices = torch.randint(max_idx, (BATCH_SIZE,))
    x = torch.stack([data[i : i + BLOCK_SIZE] for i in indices])
    y = torch.stack([data[i + 1 : i + BLOCK_SIZE + 1] for i in indices])
    return x.to(DEVICE), y.to(DEVICE)


# -----------------------------------------------------------------------------
# 2. Normalization Primitives
# -----------------------------------------------------------------------------
class RMSNorm(nn.Module):
    """Canonical RMSNorm with learnable affine scaling."""
    def __init__(self, dim: int, eps: float = 1e-6) -> None:
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        rms = torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return x * rms * self.weight


class ConicalNorm(nn.Module):
    """Radial spherical projection onto S^{d-1} without learnable affine weights."""
    def __init__(self, dim: int, eps: float = 1e-7) -> None:
        super().__init__()
        self.eps = eps
        self.scale = float(dim ** 0.5)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        norm = torch.norm(x, p=2, dim=-1, keepdim=True) + self.eps
        return self.scale * (x / norm)


class EquatorialNorm(nn.Module):
    """Zero-trace intra-token projection onto S^{d-2} without learnable affine weights."""
    def __init__(self, dim: int, eps: float = 1e-6) -> None:
        super().__init__()
        self.eps = eps
        self.scale = float(dim ** 0.5)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        u = x.mean(dim=-1, keepdim=True)
        x_centered = x - u
        norm = torch.norm(x_centered, p=2, dim=-1, keepdim=True) + self.eps
        return self.scale * (x_centered / norm)


def build_norm_layer(norm_type: str, dim: int) -> nn.Module:
    if norm_type == "layernorm":
        return nn.LayerNorm(dim)
    elif norm_type == "rmsnorm":
        return RMSNorm(dim)
    elif norm_type == "conical":
        return ConicalNorm(dim)
    elif norm_type == "equatorial":
        return EquatorialNorm(dim)
    raise ValueError(f"Unknown norm_type: {norm_type}")


# -----------------------------------------------------------------------------
# 3. Audited Causal Attention Module
# -----------------------------------------------------------------------------
class AuditedCausalSelfAttention(nn.Module):
    """
    Causal multi-head self-attention auditing logit magnitude and entropy.
    Applies spherical normalization (QK-Norm) for conical and equatorial variants.
    """
    def __init__(self, d_model: int, n_heads: int, norm_type: str = "layernorm") -> None:
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.norm_type = norm_type

        self.c_attn = nn.Linear(d_model, 3 * d_model, bias=False)
        self.c_proj = nn.Linear(d_model, d_model, bias=False)

        self.register_buffer(
            "mask",
            torch.tril(torch.ones(BLOCK_SIZE, BLOCK_SIZE)).view(1, 1, BLOCK_SIZE, BLOCK_SIZE),
        )

    def forward(self, x: torch.Tensor, record_metrics: bool = False) -> Tuple[torch.Tensor, float, float]:
        b, t, c = x.size()
        q, k, v = self.c_attn(x).split(self.d_model, dim=2)

        q = q.view(b, t, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(b, t, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(b, t, self.n_heads, self.head_dim).transpose(1, 2)

        if self.norm_type in ["conical", "equatorial"]:
            # QK-Norm: Project queries and keys onto the unit sphere S^{d_k - 1}
            q_norm = F.normalize(q, p=2, dim=-1, eps=EPS)
            k_norm = F.normalize(k, p=2, dim=-1, eps=EPS)
            raw_scores = torch.matmul(q_norm, k_norm.transpose(-2, -1)) * float(self.head_dim ** 0.5)
        else:
            raw_scores = torch.matmul(q, k.transpose(-2, -1)) * (1.0 / math.sqrt(self.head_dim))

        max_logit = 0.0
        entropy = 0.0

        if record_metrics:
            with torch.no_grad():
                valid_logits = raw_scores.masked_select(self.mask[:, :, :t, :t] == 1)
                max_logit = float(valid_logits.max().item()) if valid_logits.numel() > 0 else 0.0

        masked_scores = raw_scores.masked_fill(self.mask[:, :, :t, :t] == 0, float("-inf"))
        attn_weights = F.softmax(masked_scores, dim=-1)

        if record_metrics:
            with torch.no_grad():
                probs_clamped = attn_weights.clamp(min=1e-9)
                entropy = float(-(probs_clamped * torch.log(probs_clamped)).sum(dim=-1).mean().item())

        y = torch.matmul(attn_weights, v)
        y = y.transpose(1, 2).contiguous().view(b, t, c)
        return self.c_proj(y), entropy, max_logit


# -----------------------------------------------------------------------------
# 4. Transformer Block & NanoGPT Architecture
# -----------------------------------------------------------------------------
class AuditedTransformerBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, norm_type: str = "layernorm") -> None:
        super().__init__()
        self.norm1 = build_norm_layer(norm_type, d_model)
        self.attn = AuditedCausalSelfAttention(d_model, n_heads, norm_type=norm_type)
        self.norm2 = build_norm_layer(norm_type, d_model)

        activation = nn.SiLU() if norm_type == "equatorial" else nn.GELU()
        self.mlp = nn.Sequential(
            nn.Linear(d_model, 4 * d_model, bias=False),
            activation,
            nn.Linear(4 * d_model, d_model, bias=False),
        )

    def forward(self, x: torch.Tensor, record_metrics: bool = False) -> Tuple[torch.Tensor, float, float]:
        attn_out, entropy, max_logit = self.attn(self.norm1(x), record_metrics=record_metrics)
        x = x + attn_out
        x = x + self.mlp(self.norm2(x))
        return x, entropy, max_logit


class AuditedNanoGPT(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        d_model: int = 128,
        n_heads: int = 4,
        n_layers: int = 4,
        norm_type: str = "layernorm"
    ) -> None:
        super().__init__()
        self.norm_type = norm_type
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(BLOCK_SIZE, d_model)

        self.blocks = nn.ModuleList([
            AuditedTransformerBlock(d_model, n_heads, norm_type=norm_type)
            for _ in range(n_layers)
        ])
        self.norm_final = build_norm_layer(norm_type, d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.token_emb.weight = self.lm_head.weight

    def forward(
        self,
        idx: torch.Tensor,
        targets: torch.Tensor = None,
        record_metrics: bool = False
    ) -> Tuple[torch.Tensor, torch.Tensor, float, float]:
        b, t = idx.size()
        pos = torch.arange(0, t, dtype=torch.long, device=idx.device)
        x = self.token_emb(idx) + self.pos_emb(pos)

        total_entropy = 0.0
        max_logits_list: List[float] = []

        for block in self.blocks:
            x, ent, m_logit = block(x, record_metrics=record_metrics)
            total_entropy += ent
            max_logits_list.append(m_logit)

        avg_entropy = total_entropy / len(self.blocks)
        overall_max_logit = max(max_logits_list) if max_logits_list else 0.0

        x = self.norm_final(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))

        return logits, loss, avg_entropy, overall_max_logit


# -----------------------------------------------------------------------------
# 5. Diagnostic Evaluation Routine
# -----------------------------------------------------------------------------
@torch.no_grad()
def evaluate_model(model: nn.Module, iters: int = EVAL_ITERS) -> Tuple[float, float, float, float, bool]:
    model.eval()
    losses: List[float] = []
    entropies: List[float] = []
    max_logits: List[float] = []

    for _ in range(iters):
        xb, yb = get_batch("val")
        _, loss, entropy, m_logit = model(xb, yb, record_metrics=True)

        if torch.isnan(loss) or torch.isinf(loss) or loss.item() > 20.0:
            return float("nan"), float("nan"), 0.0, 999.0, True

        losses.append(float(loss.item()))
        entropies.append(entropy)
        max_logits.append(m_logit)

    val_loss = float(np.mean(losses))
    ppl = math.exp(min(val_loss, 15.0))
    avg_entropy = float(np.mean(entropies))
    peak_logit = float(np.max(max_logits))
    return val_loss, ppl, avg_entropy, peak_logit, False


# -----------------------------------------------------------------------------
# 6. Multi-Regime Benchmark Execution Loop
# -----------------------------------------------------------------------------
def run_benchmark() -> None:
    configs = [
        ("LayerNorm-GPT (GPT-2)", "layernorm"),
        ("RMSNorm-GPT (LLaMA)", "rmsnorm"),
        ("Conical-GPT (S^{d-1})", "conical"),
        ("Equatorial-GPT (S^{d-2})", "equatorial"),
    ]

    regimes = [
        {"name": "Regime A: Shallow Multi-Seed (L=4 | LR=1e-3)", "layers": 4, "lr": 1e-3, "steps": STEPS_NORMAL},
        {"name": "Regime B: Deep Scaling (L=8 | LR=1e-3)", "layers": 8, "lr": 1e-3, "steps": STEPS_NORMAL},
        {"name": "Regime C: High-LR Stress (L=8 | LR=3e-3)", "layers": 8, "lr": 3e-3, "steps": STEPS_STRESS},
    ]

    benchmark_storage = {
        reg["name"]: {
            cfg[0]: {"loss": [], "ppl": [], "entropy": [], "max_logit": [], "diverged": []}
            for cfg in configs
        }
        for reg in regimes
    }

    print("=" * 135)
    print(f"[INFO] Multi-Regime Transformer Robustness Benchmark | Device: {DEVICE}")
    print(f"[INFO] Seeds: {SEEDS} | Regimes: {len(regimes)} | Models: {len(configs)}")
    print("=" * 135)

    start_time = time.time()

    for reg in regimes:
        reg_name = reg["name"]
        n_layers = reg["layers"]
        lr_val = reg["lr"]
        steps = reg["steps"]

        print(f"\n[INFO] Starting {reg_name}")
        print("-" * 135)

        for name, norm_type in configs:
            for seed in SEEDS:
                torch.manual_seed(seed)
                if torch.cuda.is_available():
                    torch.cuda.manual_seed_all(seed)

                model = AuditedNanoGPT(
                    vocab_size=VOCAB_SIZE,
                    d_model=D_MODEL,
                    n_heads=N_HEADS,
                    n_layers=n_layers,
                    norm_type=norm_type,
                ).to(DEVICE)

                optimizer = torch.optim.AdamW(model.parameters(), lr=lr_val, weight_decay=1e-2)
                scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=steps)

                diverged = False
                model.train()
                for _ in range(1, steps + 1):
                    xb, yb = get_batch("train")
                    optimizer.zero_grad()
                    _, loss, _, _ = model(xb, yb)

                    if torch.isnan(loss) or torch.isinf(loss) or loss.item() > 25.0:
                        diverged = True
                        break

                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()
                    scheduler.step()

                if diverged:
                    v_loss, v_ppl, v_ent, v_logit = float("nan"), float("nan"), 0.0, 999.0
                else:
                    v_loss, v_ppl, v_ent, v_logit, eval_diverged = evaluate_model(model)
                    if eval_diverged:
                        diverged = True

                benchmark_storage[reg_name][name]["loss"].append(v_loss)
                benchmark_storage[reg_name][name]["ppl"].append(v_ppl)
                benchmark_storage[reg_name][name]["entropy"].append(v_ent)
                benchmark_storage[reg_name][name]["max_logit"].append(v_logit)
                benchmark_storage[reg_name][name]["diverged"].append(diverged)

                status_str = "DIVERGED" if diverged else f"Loss={v_loss:.4f} | PPL={v_ppl:>5.2f} | Logit={v_logit:>5.1f}"
                print(f"  [RUN] {name:<26} | Seed {seed:>4} | {status_str}")

                del model, optimizer, scheduler
                torch.cuda.empty_cache()

    total_duration = time.time() - start_time
    print(f"\n[INFO] Benchmark completed in {total_duration:.1f}s")

    # -------------------------------------------------------------------------
    # 7. Tabulated Performance Reports
    # -------------------------------------------------------------------------
    for reg in regimes:
        reg_name = reg["name"]
        print("\n" + "=" * 135)
        print(f"PERFORMANCE REPORT: {reg_name} (3 SEEDS: MEAN +/- STD)")
        print("=" * 135)
        print(
            f"{'ARCHITECTURE':<26} | {'VAL LOSS':<16} | {'PERPLEXITY':<16} | "
            f"{'ATTN ENTROPY':<16} | {'MAX QK LOGIT':<16} | {'STABILITY'}"
        )
        print("-" * 135)

        for name, _ in configs:
            records = benchmark_storage[reg_name][name]
            div_count = sum(records["diverged"])

            valid_losses = [x for x in records["loss"] if not math.isnan(x)]
            valid_ppls = [x for x in records["ppl"] if not math.isnan(x)]
            valid_ents = [x for x in records["entropy"] if not math.isnan(x)]
            valid_logits = [x for x in records["max_logit"] if not math.isnan(x)]

            if not valid_losses:
                print(f"{name:<26} | {'DIVERGED':^16} | {'--':^16} | {'--':^16} | {'--':^16} | {div_count}/3 Failed")
                continue

            l_m, l_s = np.mean(valid_losses), np.std(valid_losses)
            p_m, p_s = np.mean(valid_ppls), np.std(valid_ppls)
            e_m, e_s = np.mean(valid_ents), np.std(valid_ents)
            g_m, g_s = np.mean(valid_logits), np.std(valid_logits)

            stability_str = f"{div_count}/3 Failed" if div_count > 0 else "3/3 Stable"
            print(
                f"{name:<26} | {l_m:>6.4f} +/- {l_s:<7.4f} | {p_m:>6.2f} +/- {p_s:<7.2f} | "
                f"{e_m:>6.3f} +/- {e_s:<7.3f} | {g_m:>5.1f} +/- {g_s:<8.1f} | {stability_str}"
            )
        print("=" * 135)


if __name__ == "__main__":
    run_benchmark()

[INFO] Multi-Regime Transformer Robustness Benchmark | Device: cuda
[INFO] Seeds: [42, 1337, 2026] | Regimes: 3 | Models: 4

[INFO] Starting Regime A: Shallow Multi-Seed (L=4 | LR=1e-3)
---------------------------------------------------------------------------------------------------------------------------------------
  [RUN] LayerNorm-GPT (GPT-2)      | Seed   42 | Loss=2.3727 | PPL=10.73 | Logit= 48.8
  [RUN] LayerNorm-GPT (GPT-2)      | Seed 1337 | Loss=2.3427 | PPL=10.41 | Logit= 46.3
  [RUN] LayerNorm-GPT (GPT-2)      | Seed 2026 | Loss=2.3355 | PPL=10.33 | Logit= 32.5
  [RUN] RMSNorm-GPT (LLaMA)        | Seed   42 | Loss=2.4024 | PPL=11.05 | Logit= 44.2
  [RUN] RMSNorm-GPT (LLaMA)        | Seed 1337 | Loss=2.3566 | PPL=10.56 | Logit= 38.5
  [RUN] RMSNorm-GPT (LLaMA)        | Seed 2026 | Loss=2.3479 | PPL=10.46 | Logit= 29.0
  [RUN] Conical-GPT (S^{d-1})      | Seed   42 | Loss=2.3407 | PPL=10.39 | Logit=  5.5
  [RUN] Conical-GPT (S^{d-1})      | Seed 1337 | Loss=2.3454 | PPL=10